In [13]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [14]:
import uuid

from src.azure_sql import (
    get_engine,
    test_connection,
    read_sql,
    write_dataframe
)

from src.qa import (
    create_check,
    check_not_empty,
    check_no_nulls,
    check_unique,
    check_journals_balanced,
    results_to_dataframe,
    raise_if_critical_failures
)

In [16]:
engine = get_engine()
test_connection(engine)

run_id = str(uuid.uuid4())

print("QA Run ID:", run_id)

QA Run ID: 4d51d91d-1b34-4c0c-9159-ec64b5346d43


In [17]:
latest_batch = read_sql(
    """
    SELECT TOP 1 batch_id
    FROM bronze.qbo_accounts_raw
    ORDER BY extracted_at DESC
    """,
    engine
).loc[0, "batch_id"]


bronze_accounts = read_sql(
    f"""
    SELECT *
    FROM bronze.qbo_accounts_raw
    WHERE batch_id = '{latest_batch}'
    """,
    engine
)

bronze_customers = read_sql(
    f"""
    SELECT *
    FROM bronze.qbo_customers_raw
    WHERE batch_id = '{latest_batch}'
    """,
    engine
)

bronze_journals = read_sql(
    f"""
    SELECT *
    FROM bronze.qbo_journal_entries_raw
    WHERE batch_id = '{latest_batch}'
    """,
    engine
)

dim_account = read_sql(
    "SELECT * FROM silver.dim_account",
    engine
)

dim_customer = read_sql(
    "SELECT * FROM silver.dim_customer",
    engine
)

fact_gl = read_sql(
    "SELECT * FROM silver.fact_gl",
    engine
)

pnl = read_sql(
    "SELECT * FROM gold.pnl_monthly_actual",
    engine
)

In [18]:
checks = []

checks.append(
    check_not_empty(
        bronze_accounts,
        "bronze",
        "accounts_not_empty",
        run_id
    )
)

checks.append(
    check_not_empty(
        bronze_customers,
        "bronze",
        "customers_not_empty",
        run_id
    )
)

checks.append(
    check_not_empty(
        bronze_journals,
        "bronze",
        "journals_not_empty",
        run_id
    )
)

In [19]:
checks.append(
    check_no_nulls(
        fact_gl,
        "account_id",
        "silver",
        run_id
    )
)

checks.append(
    check_no_nulls(
        fact_gl,
        "txn_date",
        "silver",
        run_id
    )
)

checks.append(
    check_unique(
        fact_gl,
        ["journal_id", "line_id"],
        "silver",
        run_id
    )
)

checks.append(
    check_journals_balanced(
        fact_gl,
        run_id
    )
)

journal_count = fact_gl["journal_no"].nunique()

checks.append(
    create_check(
        layer="silver",
        check_name="expected_journal_count",
        passed=journal_count == 36,
        actual_value=journal_count,
        expected_value=36,
        run_id=run_id
    )
)

In [20]:
checks.append(
    check_not_empty(
        pnl,
        "gold",
        "pnl_not_empty",
        run_id
    )
)

month_count = pnl["year_month"].nunique()

checks.append(
    create_check(
        layer="gold",
        check_name="expected_month_count",
        passed=month_count == 36,
        actual_value=month_count,
        expected_value=36,
        run_id=run_id
    )
)

In [21]:
qa_results = results_to_dataframe(checks)

display(
    qa_results[
        [
            "layer",
            "check_name",
            "status",
            "severity",
            "actual_value",
            "expected_value"
        ]
    ]
)

,layer,check_name,status,severity,actual_value,expected_value
0,bronze,accounts_not_empty,PASS,ERROR,89,> 0
1,bronze,customers_not_empty,PASS,ERROR,411,> 0
2,bronze,journals_not_empty,PASS,ERROR,39,> 0
3,silver,account_id_not_null,PASS,ERROR,0,0
4,silver,txn_date_not_null,PASS,ERROR,0,0
5,silver,unique_journal_id_line_id,PASS,ERROR,0,0
6,silver,journals_balanced,PASS,ERROR,0,0
7,silver,expected_journal_count,PASS,ERROR,36,36
8,gold,pnl_not_empty,PASS,ERROR,828,> 0
9,gold,expected_month_count,PASS,ERROR,36,36


In [25]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS qa.pipeline_checks
    """))

In [27]:
from sqlalchemy import text
from sqlalchemy.dialects.mssql import NVARCHAR, DATETIMEOFFSET

qa_results["actual_value"] = qa_results["actual_value"].astype(str)
qa_results["expected_value"] = qa_results["expected_value"].astype(str)

qa_dtype = {
    "run_id": NVARCHAR(36),
    "checked_at": DATETIMEOFFSET(),
    "layer": NVARCHAR(50),
    "check_name": NVARCHAR(200),
    "status": NVARCHAR(20),
    "severity": NVARCHAR(20),
    "actual_value": NVARCHAR(200),
    "expected_value": NVARCHAR(200),
    "message": NVARCHAR(None)
}

write_dataframe(
    qa_results,
    table="pipeline_checks",
    schema="qa",
    if_exists="append",
    engine=engine,
    dtype=qa_dtype
)

write_dataframe(
    qa_results,
    table="pipeline_checks",
    schema="qa",
    if_exists="append",
    engine=engine
)

print("QA results saved.")

QA results saved.


In [28]:
raise_if_critical_failures(checks)

print("All critical QA checks passed.")

All critical QA checks passed.
